# 04 模型建模 (core.models)

基于 `hscredit.xlsx` 真实信贷数据，覆盖 LogisticRegression / ScoreCard / 损失函数 / 概率转评分。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (init_setting, OptimalBinning, WOEEncoder,
    LogisticRegression, ScoreCard,
    FocalLoss, WeightedBCELoss, KSMetric, GiniMetric,
)
from hscredit.core.metrics import ks, auc, gini
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D', origin='1899-12-30')
target = 'FPD15'
features = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','lender_count_6m',
    'overdue_loan_m1_count_12m','overdue_loan_m0_count_12m','loan_count_12m',
    'loan_amt_sum_12m','network_loan_lender_count','hit_lender_count',
    'last_performance_days','credit_loan_duration','lender_inquiry_count_3m']
X = df[features].fillna(-999); y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'训练集: {X_tr.shape}, 坏率: {y_tr.mean():.2%}')

## 1. WOE 编码

In [ ]:
woe = WOEEncoder(max_n_bins=8, method='mdlp')
X_woe_tr = woe.fit_transform(X_tr, y_tr)
X_woe_te = woe.transform(X_te)
print('WOE编码完成:', X_woe_tr.shape)

## 2. LogisticRegression（含统计量）

In [ ]:
lr = LogisticRegression(C=0.1, max_iter=1000, calculate_stats=True)
lr.fit(X_woe_tr, y_tr)
y_prob_tr = lr.predict_proba(X_woe_tr)[:,1]
y_prob_te = lr.predict_proba(X_woe_te)[:,1]
print(f'训练集  KS:{ks(y_tr,y_prob_tr):.4f}  AUC:{auc(y_tr,y_prob_tr):.4f}  Gini:{gini(y_tr,y_prob_tr):.4f}')
print(f'测试集  KS:{ks(y_te,y_prob_te):.4f}  AUC:{auc(y_te,y_prob_te):.4f}  Gini:{gini(y_te,y_prob_te):.4f}')

In [ ]:
print('\n模型系数统计摘要:')
lr.summary()

## 3. ScoreCard — 生成评分卡

In [ ]:
sc = ScoreCard(
    pdo=20, score=600, odds=1/19,
    lr_kwargs={'C': 0.1, 'max_iter': 1000, 'calculate_stats': True}
)
sc.fit(X_woe_tr, y_tr, woe_encoder=woe)
scores_te = sc.predict(X_woe_te)
print(f'评分分布:')
print(pd.Series(scores_te).describe().round(1))
print(f'\n评分区间坏率分析:')
score_bins = pd.cut(scores_te, bins=[0,500,550,580,600,620,640,660,1000])
bad_rate = y_te.values
print(pd.DataFrame({'评分区间': score_bins, '坏标': bad_rate}).groupby('评分区间')['坏标'].agg(['mean','count']).round(4))

In [ ]:
print('\n评分卡表:')
sc.scorecard_table_.head(20)

## 4. 概率转评分

In [ ]:
from hscredit.core.models.probability_to_score import probability_to_score
scores_direct = probability_to_score(y_prob_te, pdo=20, score=600, odds=1/19)
print('概率转评分对比:')
pd.DataFrame({'概率': y_prob_te[:8].round(4), '评分(ScoreCard)': scores_te[:8], '评分(直接转换)': scores_direct[:8].round(0)})

## 5. 自定义损失函数

In [ ]:
# 信贷场景：正负样本不均衡，使用FocalLoss减少易分类样本权重
print(f'样本不均衡比: {(y_tr==0).sum()}:{(y_tr==1).sum()} = {(y_tr==0).sum()/(y_tr==1).sum():.1f}:1')

fl = FocalLoss(gamma=2.0, alpha=0.25)
print('FocalLoss:', fl)

wbce = WeightedBCELoss(pos_weight=float((y_tr==0).sum()/(y_tr==1).sum()))
print('WeightedBCE pos_weight:', wbce)

print('KSMetric:', KSMetric())
print('GiniMetric:', GiniMetric())

## 6. 系数权重可视化

In [ ]:
from hscredit import plot_weights
fig = plot_weights(lr)
plt.show()